In [18]:
import numpy as np
import pandas as pd
import models
import json
import re
from datetime import datetime
from database import engine, sessionDB
from nba_api.stats.static import players
from nba_api.stats.endpoints import playerawards, commonplayerinfo, playercareerstats
from langchain_ollama import ChatOllama
from decouple import config
from ddgs import DDGS

In [19]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [20]:
players= players.get_players()
#players

In [21]:
models.Base.metadata.create_all(bind=engine)
db = sessionDB()

player testing and structure allat stuff

In [22]:
#using lebron cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
pid = lebron["id"] #2544
info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

needed_columns = [
    "DISPLAY_FIRST_LAST", 
    "BIRTHDATE", 
    "COUNTRY", 
    "HEIGHT", 
    "WEIGHT", 
    "DRAFT_YEAR", 
    "DRAFT_ROUND", 
    "DRAFT_NUMBER"
]

#nicknames and school i need to use an agent for

info_df = info_df[needed_columns]
info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)

info_df

,NAME,BIRTHDATE,COUNTRY,HEIGHT,WEIGHT,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,LeBron James,1984-12-30T00:00:00,USA,6-9,250,2003,1,1


actual player info parsing

In [23]:
#height to cm

def to_CM(height: str):
    height = height.split("-")
    return round(int(height[0]) * 30.48 + int(height[1]) * 2.54)

to_CM("6-9")

206

In [24]:
#agent to get nicknames + school

def search(query: str) -> str:
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    if not results:
        return "no results found"
    return "\n\n".join(
        f"Title: {r['title']}\nSnippet: {r['body']}"
        for r in results
    )

llm = ChatOllama(
    model=config("OLLAMA_MODEL", default="qwen2.5:7b-instruct"), 
    base_url=config("OLLAMA_BASE_URL", "http://localhost:11434/"),
    temperature=0
)

def getExtraPlayerInfo(player_name: str) -> dict:
    search_results = search(f"{player_name} NBA nicknames college university")

    prompt = f"""Based on these search results about NBA player {player_name}:

    {search_results}

    Return ONLY a valid JSON object in this exact format:
    {{"nicknames": ["nickname1", "nickname2"], "college": "University Name or null"}}

    Rules:
    - college should be null if they went straight from high school
    - nicknames should be an empty list if none are found
    """

    response = llm.invoke(prompt)
    text = response.content

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f"Could not parse response: {text}")


In [25]:
def getPlayer(pid: int):
    info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

    needed_columns = [
        "DISPLAY_FIRST_LAST", 
        "BIRTHDATE", 
        "COUNTRY", 
        "HEIGHT", 
        "WEIGHT", 
        "DRAFT_YEAR", 
        "DRAFT_ROUND", 
        "DRAFT_NUMBER"
    ]

    #nicknames and school i need to use an agent for

    info_df = info_df[needed_columns]
    info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)
    return info_df

bron_id = 2544

info_df = getPlayer(bron_id)
player_name = info_df["NAME"].iloc[0]

extra_info = None
while not extra_info:
    extra_info = getExtraPlayerInfo(player_name)

print(extra_info)

pinfo = models.Player(
    player_id = bron_id, #dont hardcode this in prod
    name = info_df["NAME"].iloc[0],
    nicknames = extra_info['nicknames'],
    country = info_df["COUNTRY"].iloc[0],
    school = extra_info['college'],
    birthdate = datetime.fromisoformat(info_df["BIRTHDATE"].iloc[0]).date(),
    height = to_CM(info_df["HEIGHT"].iloc[0]),
    weight = int(info_df["WEIGHT"].iloc[0]),
    draft_year = int(info_df["DRAFT_YEAR"].iloc[0]),
    draft_round = int(info_df["DRAFT_ROUND"].iloc[0]),
    draft_pick = int(info_df["DRAFT_NUMBER"].iloc[0]),
)

#db.rollback()
#db.add(pinfo)
#db.commit()
#db.close()


{'nicknames': ['King James'], 'college': None}


award testing and structure allat stuff

In [26]:
#using lebron again cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544
lebron_awards = playerawards.PlayerAwards(player_id=lebron_id).get_data_frames()[0]

data = lebron_awards

acronym_hmap = {
    "All-Defensive Team": "DEF",''
    "All-NBA": "NBA",
    "All-Rookie Team": "ROOK",
    "NBA All-Star": "AS",
    "NBA All-Star Most Valuable Player": "ASMVP",
    "NBA Champion": "CHAMP",
    "NBA Cup All-Tournament Team": "CUP",
    "NBA Cup Most Valuable Player": "CUPMVP",
    "NBA Finals Most Valuable Player": "FMVP",
    "NBA Most Valuable Player": "MVP",
    "NBA Player of the Month": "POTM",
    "NBA Player of the Week": "POTW",
    "NBA Rookie of the Month": "ROTM",
    "NBA Rookie of the Year": "ROTY",
    "NBA Defensive Player of the Year": "DPOY",
    "NBA Defensive Player of the Month": "DPOM",
    "NBA Most Improved Player": "MIP",
    "NBA Clutch Player of the Year": "CPOY",
    "NBA Sportsmanship": "SPTMAN",
}

all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

#gotta add stat champs + cfmvp + hustle player oty + teammate oty + dunk/3pt

clean_lebron_awards_df = lebron_awards[lebron_awards["DESCRIPTION"].isin(acronym_hmap)]
clean_lebron_awards_df = clean_lebron_awards_df.drop([
    "FIRST_NAME", 
    "LAST_NAME", 
    "CONFERENCE", 
    "TYPE", 
    "SUBTYPE1", 
    "SUBTYPE2", 
    "SUBTYPE3",
    "MONTH",
    "WEEK",
    "PERSON_ID",
    "TEAM"
], axis=1)

clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"] = clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
clean_lebron_awards_df = clean_lebron_awards_df.replace(r'^\s*$', 0, regex=True) 
#^ gets rid of empty strings

clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df["DESCRIPTION"].map(acronym_hmap)
clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df.apply(
    lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
    if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
    axis=1
)

clean_lebron_awards_df
#data

,DESCRIPTION,ALL_NBA_TEAM_NUMBER,SEASON
0,DEF1,1,2008-09
1,DEF1,1,2009-10
2,DEF1,1,2010-11
3,DEF1,1,2011-12
4,DEF1,1,2012-13
5,DEF2,2,2013-14
6,NBA2,2,2004-05
7,NBA1,1,2005-06
8,NBA2,2,2006-07
9,NBA1,1,2007-08


actual award parsing

In [27]:
def getAwards(pid: int):
    awards_df = playerawards.PlayerAwards(player_id=pid).get_data_frames()[0]

    acronym_hmap = {
        "All-Defensive Team": "DEF",''
        "All-NBA": "NBA",
        "All-Rookie Team": "ROOK",
        "NBA All-Star": "AS",
        "NBA All-Star Most Valuable Player": "ASMVP",
        "NBA Champion": "CHAMP",
        "NBA Cup All-Tournament Team": "CUP",
        "NBA Cup Most Valuable Player": "CUPMVP",
        "NBA Finals Most Valuable Player": "FMVP",
        "NBA Most Valuable Player": "MVP",
        "NBA Player of the Month": "POTM",
        "NBA Player of the Week": "POTW",
        "NBA Rookie of the Month": "ROTM",
        "NBA Rookie of the Year": "ROTY",
    }

    all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

    awards_df = awards_df[awards_df["DESCRIPTION"].isin(acronym_hmap)]
    awards_df = awards_df.drop([
        "FIRST_NAME", 
        "LAST_NAME", 
        "CONFERENCE", 
        "TYPE", 
        "SUBTYPE1", 
        "SUBTYPE2", 
        "SUBTYPE3",
        "MONTH",
        "WEEK",
        "PERSON_ID",
        "TEAM"
    ], axis=1)

    awards_df["ALL_NBA_TEAM_NUMBER"] = awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
    awards_df = awards_df.replace(r'^\s*$', 0, regex=True) 
    #^ gets rid of empty strings

    awards_df["DESCRIPTION"] = awards_df["DESCRIPTION"].map(acronym_hmap)
    awards_df["DESCRIPTION"] = awards_df.apply(
        lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
        if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
        axis=1
    )

    return awards_df.drop(["ALL_NBA_TEAM_NUMBER"], axis=1)

bron_id = 2544

awards = getAwards(bron_id)

award_list = [
    models.Award(
        player_id = bron_id, #dont hardcode this in prod
        season = award.SEASON,
        award_name = award.DESCRIPTION
    )
    for award in awards.itertuples()
]

db.rollback()
db.add_all(award_list)
db.commit()
db.close()

season stat testing

In [28]:
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544

